***Read CSV Files***

In [0]:
%run "/Workspace/Users/training@nityacloudtech.onmicrosoft.com/NityaCloudTech_Solution_Architect/tools/env_setup"

In [0]:
source_path =f"/Volumes/{prefix}_account/stage/employee/Inbound"

Catalog_account=f"{prefix}_account"

# display(dbutils.fs.ls(source_path))
if dbutils.fs.ls(source_path):
    print("New records to process")
else:
    dbutils.notebook.exit("No new records to process")

***LAst Modifed Date From Dim_employee table***

In [0]:
from pyspark.sql.functions import max

max_last_mdofied_date=spark.table(f"{Catalog_account}.{bronze_schema}.Dim_employees").agg(max("last_modified_date").alias("Max_Employees")).collect()[0]["Max_Employees"]
last_mdofied_date =max_last_mdofied_date if max_last_mdofied_date is not None else "2025-01-01"

print(last_mdofied_date)

In [0]:
df_employees = spark.read.format("csv").option("header","true").option("inferSchema","true").load(source_path)


In [0]:
df_inc_employees=df_employees.filter(f"last_modified_date > '{last_mdofied_date}'")



In [0]:
if df_inc_employees.count() == 0:
    dbutils.notebook.exit("No new records to process")

***Audit Fields***

In [0]:
from pyspark.sql.functions import lit,current_timestamp,current_date,current_timestamp,current_date,current_timestamp,current_date,current_timestamp,current_date,current_timestamp,current_user
# select select expr expr withcolumn and withcolumnrenamed

df_audit=df_employees.withColumn("CreatedBy",lit(current_user()))\
    .withColumn("CreatedDate",current_timestamp())\
    .withColumn("UpdatedBy",lit(current_user()))\
    .withColumn("UpdatedDate",current_timestamp())\
    .withColumn("last_modified_date",current_timestamp())

In [0]:
from pyspark.sql.functions import col
from delta.tables import *

# df_target=spark.table("dev_account.bronze.dim_employees").schema


df_dim_account_schema=spark.table("dev_account.bronze.dim_employees").schema

df_select=df_audit.select(
    [col(field.name).cast(field.dataType) for field in df_dim_account_schema]
)
    

In [0]:

dim_account_tgt=DeltaTable.forName(spark,"dev_account.bronze.dim_employees")

merge_condition="(tgt.employee_id = src.employee_id and tgt.is_active = true)"

update_condition = ("tgt.employee_id = src.employee_id" and "tgt.is_active = true" and 
    (("tgt.First_Name <> src.first_name ")
    or ("tgt.Last_name <> src.last_name ")
    or ("tgt.department <> src.department ")
    or ("tgt.job_title <> src.job_title ")
    or ("tgt.salary <> src.salary")
    or ("tgt.city <> src.city")))

merge_updates = { 
    "First_Name": "src.first_name",
    "Last_name": "src.last_name",
    "department": "src.department",
    "job_title": "src.job_title",
    "salary": "src.salary",
    "city": "src.city",
    "is_active": "src.is_active",
    "joining_date": "src.joining_date",
    "UpdatedBy": "src.UpdatedBy",
    "UpdatedDate": "src.UpdatedDate"
}

dim_account_tgt.alias("tgt").merge(
    df_select.alias("src"),
    merge_condition
).whenMatchedUpdate(set=merge_updates, condition=update_condition).whenNotMatchedInsertAll().execute()
        


In [0]:
# Move the files from Inbound to Archive folder

Source_Path = f"/Volumes/{prefix}_account/stage/employee/Inbound/"
Archive_Path = f"/Volumes/{prefix}_account/stage/employee/Archive/"

files = dbutils.fs.ls(Source_Path)

for f in files:
    if f.name.endswith(".csv"):
        dbutils.fs.mv(f.path, Archive_Path + f.name)
        # display(f.path, Archive_Path + f.name)
        print(f"Moved {f.name} to Archive")

print("All files moved to Archive successfully.")